# Agentic AI - SQL Generation with Reflection

How the reflection pattern can improve an agentic workflow that converts natural language questions to database queries written in SQL. You’ll see how an agent can spot issues in its own outputs, refine them, and improve it’s response before giving a final answer.

One Table Only

### Load libraries

In [1]:
import json
import re
import pandas as pd
import utils
from dotenv import load_dotenv
import aisuite as ai

load_dotenv()

True

### Create aisuite Client

In [2]:
client = ai.Client()

### Setup the database

Create SQLLite database - products.db

In [3]:
utils.create_transactions_db()

SQLite database 'products.db' created with a single 'transactions' table (event-sourced).


### Inspect Schema

In [4]:
utils.print_html(utils.get_schema('products.db'))

### Use LLM to query database

In [5]:
def generate_sql(question: str, schema: str, model: str) -> str:
    prompt = f"""
    You are a SQL assistant. Given the schema and the user's question, write a SQL query for SQLite.

    Schema:
    {schema}

    User question:
    {question}

    Respond with the SQL only.
    """
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "user", "content": prompt}
        ],
        temperature=0
    )
    sql_query = response.choices[0].message.content.strip()
    return sql_query

In [6]:
schema = utils.get_schema('products.db')

# We ask a question about the data in natural language
question = "Which color of product has the highest total sales?"

utils.print_html(question, title="User Question")
sql_query_v1 = generate_sql(question, schema, model="openai:gpt-4.1")
utils.print_html(sql_query_v1, title="Generated SQL Query")

### run the sql query returned

In [7]:
df_sql_V1 = utils.execute_sql(sql_query_v1, 'products.db')

utils.print_html(df_sql_V1, title="Output of SQL Query V1 - ❌ Does NOT fully answer the question")

color,total_sales
blue,-190571.46


### Improve query with reflection

In [8]:
def refine_sql(question: str, schema: str, previous_sql: str, model: str) -> tuple[str, str]:
    prompt = f"""
    You are a SQL reviewer and refiner.

    User asked:
    {question}

    Original SQL query:
    {previous_sql}

    Schema:
    {schema}

    Step 1: Briefly evaluate if the SQL OUTPUT fully answers the user's question.
    Step 2: If improvement is needed, provide a refined SQL query for SQLite.
    If the original SQL is already correct, return it unchanged.

    Return STRICT JSON with two fields:
    {{
      "feedback": "<1-3 sentences explaining the gap or confirming correctness>",
      "refined_sql": "<final SQL to run>"
    }}
    """
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "user", "content": prompt}
        ],
        temperature=0
    )
    content = response.choices[0].message.content

    try:
        result = json.loads(content)
        feedback = result.get("feedback", "").strip()
        sql_query = result.get("refined_sql", "").strip()
    except json.JSONDecodeError:
        feedback = "Failed to parse JSON response."
        sql_query = previous_sql  # Fallback to previous SQL

    return feedback, sql_query

In [9]:
feedback, sql_query_v2 = refine_sql(question, schema, sql_query_v1, model="openai:gpt-4.1")
utils.print_html(feedback, title="Refinement Feedback")
utils.print_html(sql_query_v2, title="Refined SQL Query V2")

# Execute and show V1 output
df_sql_V1 = utils.execute_sql(sql_query_v1, db_path='products.db')
utils.print_html(df_sql_V1, title="SQL Output of V1 - ❌ Does NOT fully answer the question")

# --- Feedback + V2 ---
utils.print_html(feedback, title="Feedback on V1")
utils.print_html(sql_query_v2, title="Refined SQL Query (V2)")

# Execute and show V2 output
df_sql_V2 = utils.execute_sql(sql_query_v2, db_path='products.db')
utils.print_html(df_sql_V2, title="SQL Output of V2 - ❌ Does NOT fully answer the question")

color,total_sales
blue,-190571.46


color,total_sales
blue,-190571.46


In [14]:
def refine_sql_with_feedback(question: str, schema: str, previous_sql: str, df_feedback: pd.DataFrame, model: str) -> tuple[str, str]:
    prompt = f"""
    You are a SQL reviewer and refiner.

    User asked:
    {question}

    Original SQL:
    {previous_sql}

    Schema:
    {schema}

    SQL Output:
    {df_feedback.to_markdown(index=False)}

    Step 1: Briefly evaluate if the SQL output answers the user's question.
    Step 2: If the SQL could be improved, provide a refined SQL query.
    If the original SQL is already correct, return it unchanged.

    Return a strict JSON object with two fields:
    - "feedback": brief evaluation and suggestions
    - "refined_sql": the final SQL to run
    """

    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=1.0,
    )

    
    content = response.choices[0].message.content
    utils.print_html(content, title="Raw Model Response for SQL Refinement")
    clean_content = re.sub(r"^```json|```$", "", content.strip(), flags=re.MULTILINE).strip()
    try:
        obj = json.loads(clean_content)
        feedback = str(obj.get("feedback", "")).strip()
        refined_sql = str(obj.get("refined_sql", previous_sql)).strip()
        if not refined_sql:
            refined_sql = previous_sql
    except Exception:
        # Fallback if the model does not return valid JSON:
        # use the raw content as feedback and keep the original SQL
        feedback = clean_content.strip()
        refined_sql = previous_sql

    return feedback, refined_sql  

In [11]:
# Use external feedback to evaluate and refine
feedback, sql_V2 = refine_sql_with_feedback(
    question=question,
    previous_sql=sql_query_v1,   # V1 query
    df_feedback=df_sql_V1,    # Output of V1
    schema=schema,
    model="openai:gpt-4.1"
)

# --- V1 ---
utils.print_html(question, title="User Question")
utils.print_html(sql_query_v1, title="Generated SQL Query (V1)")
utils.print_html(df_sql_V1, title="SQL Output of V1 - ❌ Does NOT fully answer the question")

# --- Feedback & V2 ---
utils.print_html(feedback, title="Feedback on V1")
utils.print_html(sql_V2, title="Refined SQL Query (V2)")

# Execute and display V2 results
df_sql_V2 = utils.execute_sql(sql_V2, db_path='products.db')
utils.print_html(df_sql_V2, title="SQL Output of V2 (with External Feedback) - ✅ Fully answers the question")


color,total_sales
blue,-190571.46


color,total_sales
white,358315.09


### Buld workflow pipeline

1. Extract database schema
2. Generate initial V1 Sql Query
3. Reflect on V1 using execution feedback
4. Execute the improved V2 Sql Query

In [12]:
def run_sql_workflow(
    db_path: str,
    question: str,
    model_generation: str = "openai:gpt-4.1",
    model_evaluation: str = "openai:gpt-4.1",
):
    """
    End-to-end workflow to generate, execute, evaluate, and refine SQL queries.

    Steps:
      1) Extract database schema
      2) Generate SQL (V1)
      3) Execute V1 → show output
      4) Reflect on V1 with execution feedback → propose refined SQL (V2)
      5) Execute V2 → show final answer
    """

    # Step 1: Extract schema
    schema = utils.get_schema(db_path)
    utils.print_html(
        schema,
        title="📘 Step 1 — Extract Database Schema"
    )
    
    # Step 2: Generate initial SQL query (V1)
    sql_query_v1 = generate_sql(question, schema, model_generation)
    utils.print_html(
        sql_query_v1,
        title="📝 Step 2 — Generated SQL Query (V1)"
    )

    # Step 3: Execute V1 and get feedback
    df_sql_v1 = utils.execute_sql(sql_query_v1, db_path)
    utils.print_html(
        df_sql_v1,
        title="🚀 Step 3 — SQL Output of V1"
    )

    # Step 4: Refine SQL using feedback
    feedback, sql_query_v2 = refine_sql_with_feedback(
        question=question,
        previous_sql=sql_query_v1,
        df_feedback=df_sql_v1,
        schema=schema,
        model=model_evaluation
    )
    utils.print_html(
        feedback,
        title="🔍 Step 4 — Feedback on V1"
    )
    utils.print_html(
        sql_query_v2,
        title="✍️ Step 4 — Refined SQL Query (V2)"
    )

    # Step 5: Execute refined SQL (V2)
    df_sql_v2 = utils.execute_sql(sql_query_v2, db_path)
    utils.print_html(
        df_sql_v2,
        title="🏁 Step 5 — SQL Output of V2"
    )

In [17]:
run_sql_workflow(
    "products.db", 
    "Which color of product has the highest total sales and returns? also add total sales amount and number of products. negative qty_delta is for product returnedbuild query which shows the returns and sales.",
    model_generation="openai:gpt-4o",
    model_evaluation="openai:gpt-4o"
)

color,total_sales_amount,total_returns_amount,total_number_of_products
white,68437.36,-358315.09,9754


color,total_sales_amount,total_returns_amount,total_number_of_products
blue,40060.28,-190571.46,5356
